In [1]:
# --- headless execution harness (added by run_experiments.py) ---
import os as _h_os
import matplotlib as _h_mpl
_h_mpl.use('Agg')
import matplotlib.pyplot as _h_plt
_OUT = 'C:\\Users\\ASUS\\group_sgu26_deep_learning\\BTL03\\output\\2. CNN + MLP\\2_Human_Gender_CNN_pytorch'
_h_os.makedirs(_OUT, exist_ok=True)
_h_n = [0]
_h_show = _h_plt.show
def _saving_show(*args, **kwargs):
    _figs = [_h_plt.figure(n) for n in _h_plt.get_fignums()]
    for _f in _figs:
        _h_n[0] += 1
        _f.savefig(_h_os.path.join(_OUT, 'fig_%02d.png' % _h_n[0]), dpi=110, bbox_inches='tight')
    _h_show(*args, **kwargs)
    for _f in _figs:
        _h_plt.close(_f)
_h_plt.show = _saving_show
print('[harness] figures -> ' + _OUT)


[harness] figures -> C:\Users\ASUS\group_sgu26_deep_learning\BTL03\output\2. CNN + MLP\2_Human_Gender_CNN_pytorch


In [2]:
# Demo (PyTorch)

# Overview
- WIKI dataset (face images + gender + age + ...)
  + Goal: classifying human gender
  + Input: a 64x64 image (PyTorch format: `(1, 64, 64)` - NCHW)
  + Output: gender (female or male)
- Defining a CNN+MLP for WIKI dataset in PyTorch
- Training and evaluating a CNN+MLP

# Import libraries

In [3]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [4]:
num_classes = 2   # male vs. female

# (1) Data

### Download data at https://drive.google.com/drive/folders/0BxINLo5jshCRYW8xODhNSlkyLTQ?resourcekey=0-RNJvvSSE67Q1bMqQi3A1RQ&usp=sharing
This source code and `.npy` files MUST be in the same location

In [5]:
x_train = np.load('64_64_11938_4098_train_x_onehot.npy')
y_train_oh = np.load('64_64_11938_4098_train_y_onehot.npy')
# .copy() bắt buộc vì np.rot90 tạo negative-stride view mà torch.from_numpy không hỗ trợ trực tiếp
x_train = np.rot90(x_train, k=3, axes=(1, 2)).copy()

x_test = np.load('64_64_5968_4098_val_x_onehot.npy')
y_test_oh = np.load('64_64_5968_4098_val_y_onehot.npy')
x_test = np.rot90(x_test, k=3, axes=(1, 2)).copy()

# Chuyển NHWC (N, 64, 64, 1) -> NCHW (N, 1, 64, 64) cho PyTorch Conv2d
if x_train.ndim == 3:
    x_train = np.expand_dims(x_train, axis=1)
    x_test = np.expand_dims(x_test, axis=1)
else:
    x_train = np.transpose(x_train, (0, 3, 1, 2)).copy()
    x_test = np.transpose(x_test, (0, 3, 1, 2)).copy()

x_train = x_train.astype(np.float32)
x_test = x_test.astype(np.float32)

# Chuyển one-hot -> nhãn số nguyên (0 hoặc 1) cho nn.CrossEntropyLoss
y_train = np.argmax(y_train_oh, axis=1).astype(np.int64)
y_test = np.argmax(y_test_oh, axis=1).astype(np.int64)

print('x_train:', x_train.shape, '| y_train:', y_train.shape)
print('x_test :', x_test.shape,  '| y_test :', y_test.shape)

x_train: (11938, 1, 64, 64) | y_train: (11938,)
x_test : (5968, 1, 64, 64) | y_test : (5968,)


# (2) Declare model

In [6]:
class HumanGenderCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Block 1: (1, 64, 64) -> (16, 32, 32)
        self.conv1_1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv_dropout1 = nn.Dropout(0.2)

        # Block 2: (16, 32, 32) -> (32, 16, 16)
        self.conv2_1 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.conv2_3 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv_dropout2 = nn.Dropout(0.2)

        # Block 3: (32, 16, 16) -> (64, 8, 8)
        self.conv3_1 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.conv_dropout3 = nn.Dropout(0.2)

        # Block 4: (64, 8, 8) -> (128, 4, 4)
        self.conv4_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(2, 2)
        self.conv_dropout4 = nn.Dropout(0.2)

        # Block 5: (128, 4, 4) -> (256, 2, 2)
        self.conv5_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(2, 2)
        self.conv_dropout5 = nn.Dropout(0.2)

        # Flatten: 256 * 2 * 2 = 1024
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 2 * 2, 1024)
        self.fc_dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(1024, 1024)
        self.fc_dropout2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.conv_dropout1(self.pool1(x))

        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = F.relu(self.conv2_3(x))
        x = self.conv_dropout2(self.pool2(x))

        x = F.relu(self.conv3_1(x))
        x = F.relu(self.conv3_2(x))
        x = F.relu(self.conv3_3(x))
        x = self.conv_dropout3(self.pool3(x))

        x = F.relu(self.conv4_1(x))
        x = F.relu(self.conv4_2(x))
        x = F.relu(self.conv4_3(x))
        x = self.conv_dropout4(self.pool4(x))

        x = F.relu(self.conv5_1(x))
        x = F.relu(self.conv5_2(x))
        x = F.relu(self.conv5_3(x))
        x = self.conv_dropout5(self.pool5(x))

        x = self.flatten(x)
        x = self.fc_dropout1(F.relu(self.fc1(x)))
        x = self.fc_dropout2(F.relu(self.fc2(x)))
        return self.fc3(x)

model = HumanGenderCNN(num_classes=num_classes).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_params:,}')  # Khớp chính xác 4,063,570 params của bản Keras

HumanGenderCNN(
  (conv1_1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1_2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout1): Dropout(p=0.2, inplace=False)
  (conv2_1): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2_2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2_3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout2): Dropout(p=0.2, inplace=False)
  (conv3_1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3_2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3_3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=Fals

In [7]:
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, 1, 64, 64), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

Layer (type:depth-idx)                   Output Shape              Param #
HumanGenderCNN                           [1, 2]                    --
├─Conv2d: 1-1                            [1, 16, 64, 64]           160
├─Conv2d: 1-2                            [1, 16, 64, 64]           2,320
├─MaxPool2d: 1-3                         [1, 16, 32, 32]           --
├─Dropout: 1-4                           [1, 16, 32, 32]           --
├─Conv2d: 1-5                            [1, 32, 32, 32]           4,640
├─Conv2d: 1-6                            [1, 32, 32, 32]           9,248
├─Conv2d: 1-7                            [1, 32, 32, 32]           9,248
├─MaxPool2d: 1-8                         [1, 32, 16, 16]           --
├─Dropout: 1-9                           [1, 32, 16, 16]           --
├─Conv2d: 1-10                           [1, 64, 16, 16]           18,496
├─Conv2d: 1-11                           [1, 64, 16, 16]           36,928
├─Conv2d: 1-12                           [1, 64, 16, 16]        

# (3) Train defined model

In [8]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            correct += (logits.argmax(dim=1) == by).sum().item()
            total += bx.size(0)
    return total_loss / total, correct / total


lr = 0.0001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

batch_size = 128
epochs = 30

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=batch_size, shuffle=False
)

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

starting_time = time.time()
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(dim=1) == by).sum().item()
        total += bx.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f'Epoch {epoch:2d}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')

print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

Epoch  1/30 - loss: 0.5483 - acc: 0.7789 - val_loss: 0.5120 - val_acc: 0.7885


Epoch  2/30 - loss: 0.5125 - acc: 0.7885 - val_loss: 0.5139 - val_acc: 0.7885


Epoch  3/30 - loss: 0.5097 - acc: 0.7885 - val_loss: 0.5096 - val_acc: 0.7885


Epoch  4/30 - loss: 0.5006 - acc: 0.7885 - val_loss: 0.4906 - val_acc: 0.7885


Epoch  5/30 - loss: 0.4863 - acc: 0.7891 - val_loss: 0.5002 - val_acc: 0.7885


Epoch  6/30 - loss: 0.4816 - acc: 0.7913 - val_loss: 0.4855 - val_acc: 0.7956


... (20 outputs left out)

Epoch 27/30 - loss: 0.2828 - acc: 0.8896 - val_loss: 0.3980 - val_acc: 0.8435


Epoch 28/30 - loss: 0.2664 - acc: 0.8973 - val_loss: 0.4262 - val_acc: 0.8413


Epoch 29/30 - loss: 0.2608 - acc: 0.8947 - val_loss: 0.3952 - val_acc: 0.8465


Epoch 30/30 - loss: 0.2453 - acc: 0.9063 - val_loss: 0.4113 - val_acc: 0.8465
> training time is 2.3610 minutes


# (4) Evaluate trained model

In [9]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

Test loss: 0.41126360490558617
Test accuracy: 0.846514745308311


In [10]:
def int2gender(num):
    return 'female' if num == 0 else 'male'

In [11]:
# select a test image randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]   # shape: (1, 64, 64)
test_label = int(y_test[random_test_index])

model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    pred_label = int(model(inp).argmax(dim=1).item())

plt.imshow(test_img[0], cmap='gray')
plt.title('true label = %s, predicted label = %s' % (int2gender(test_label), int2gender(pred_label)))
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_28384\3836801577.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)
